### Agent ReAct Docstore - Practice & Exercises

In this notebook, you will practice integrating RAG pipelines as query tools inside ReAct agents.

#### Part 1: Follow-Along Practice

##### Goal / Question
Set up a RAG pipeline querying a local database, wrap it inside a custom tool, and bind it to a ReAct agent to answer domain-specific questions.

##### Sample Input
```python
{"input": "What is Apple Intelligence?"}
```

##### Sample Output
A response containing retrieved information from document store database.

##### Plan / Steps
1. Load existing Chroma DB store and configure embeddings using `GoogleGenAIEmbeddings(model="models/text-embedding-004")`.
2. Establish history-aware retrievers and document QA retrieval chain.
3. Expose retrieval chain as a `Tool` constructor.
4. Pull ReAct agent prompt `hwchase17/react` from hub.
5. Initialize model using `init_chat_model("llama-3.3-70b-versatile", model_provider="groq")`.
6. Build agent executor and run query.

In [ ]:
import os
from dotenv import load_dotenv
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import Tool
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenAIEmbeddings

load_dotenv()
persistent_directory = "../../4_rag/db/chroma_db_with_metadata"

if os.path.exists(persistent_directory):
    embeddings = GoogleGenAIEmbeddings(model="models/text-embedding-004")
    db = Chroma(persist_directory=persistent_directory, embedding_function=embeddings)
    retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
    
    context_prompt = ChatPromptTemplate.from_messages([
        ("system", "Contextualize the query..."),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ])
    history_aware = create_history_aware_retriever(llm, retriever, context_prompt)
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer questions based on retrieved context: {context}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ])
    qa_chain = create_stuff_documents_chain(llm, qa_prompt)
    rag_chain = create_retrieval_chain(history_aware, qa_chain)
    
    tools = [
        Tool(
            name="Answer Question",
            func=lambda q, **kwargs: rag_chain.invoke({"input": q, "chat_history": kwargs.get("chat_history", [])}),
            description="Useful for answering contextual database questions."
        )
    ]
    prompt = hub.pull("hwchase17/react")
    agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
    executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, handle_parsing_errors=True, verbose=True)
    print("Docstore retriever loaded successfully.")
else:
    print("Skipping execution: database directory not found.")

#### Part 2: Practice Exercises

##### Exercise: Dual-Source Agent

##### Goal / Question
Expand the docstore agent by adding a backup Wikipedia tool. The agent should first try to answer from the database (RAG) and fallback to Wikipedia if the information is not present in local documents.

##### Sample Input
```python
{"input": "Who is Alan Turing?"}
```

##### Sample Output
A response about Alan Turing sourced from Wikipedia.

##### Plan / Steps
1. Define a Wikipedia query function and wrap it inside a `Tool` constructor named `Wikipedia`.
2. Expose both `Answer Question` (RAG) and `Wikipedia` tools to the tools list.
3. Recreate the ReAct agent containing both tools and execute it with a non-context query.

In [ ]:
# TODO: Write your implementation here

<details>
<summary>💡 Click to see solution</summary>

```python
from wikipedia import summary
def search_wiki(query):
    try:
        return summary(query, sentences=2)
    except:
        return "No wiki page found."

wiki_tool = Tool(name="Wikipedia", func=search_wiki, description="Useful for general world facts.")
dual_tools = tools + [wiki_tool]
dual_agent = create_react_agent(llm=llm, tools=dual_tools, prompt=prompt)
dual_executor = AgentExecutor.from_agent_and_tools(agent=dual_agent, tools=dual_tools, handle_parsing_errors=True, verbose=True)
print(dual_executor.invoke({"input": "Who is Alan Turing?", "chat_history": []}))
```
</details>